# 实践 5.4  BSA设计敏感性：池大小×深度×窗口

《统计遗传与智能育种学》第五章 · 电子资源 `05d_bsa_design.ipynb`

本 notebook 的代码与书中“完整Python代码”逐字一致。运行检查：重启内核并运行全部单元（Restart Kernel and Run All）。

电子资源：05d_bsa_design.ipynb。实践固定一个真实QTL，分别改变每池个体数、平均测序深度和窗口大小；每种条件重复模拟，统计峰位置到真实QTL的偏离、主峰连续半峰高区间宽度和峰值信噪比。该简化模拟在每个位点独立抽取池频率，再模拟读段抽样，不显式生成染色体重组，因此用于分解个体抽样、读段抽样和窗口平滑三类因素的影响，不能据此估计受真实重组限制的定位分辨率或给出普适的群体规模阈值。

设计因素：pool_n × depth × window_snps  
结果：定位偏离、主峰连续半峰高区间宽度、signal/noise  
验收：至少重复20次并汇总均值，并补算离散程度；固定随机种子；解释该模拟为何不能替代显式重组群体设计。

## 输入、完整代码、实际输出与解释

输入与可复现条件：固定QTL，pool_n=20/50/120，depth=15/40/100，window=100/300/700 kb；SEED=504；每个条件重复20次模拟。

In [1]:
import sys, platform
import numpy as np
import pandas as pd
import sklearn
SEED=504
rng=np.random.default_rng(SEED)
print('Python',sys.version.split()[0])
print('Platform',platform.platform())
print('numpy',np.__version__,'pandas',pd.__version__,'scikit-learn',sklearn.__version__)
print('SEED',SEED)


Python 3.12.13
Platform macOS-26.5.2-arm64-arm-64bit
numpy 2.3.5 pandas 2.2.3 scikit-learn 1.8.0
SEED 504


In [2]:
qtl=5_000_000; pos=np.arange(500)*20000
def one(pool_n,depth,window):
 base=.5*np.ones(len(pos)); true=.28*np.exp(-((pos-qtl)/650000)**2); # 池个体抽样：先从2N等位基因抽频率
 p_hi=rng.binomial(2*pool_n,np.clip(base+true,0,1))/(2*pool_n); p_lo=rng.binomial(2*pool_n,np.clip(base-true,0,1))/(2*pool_n); dh=rng.poisson(depth,len(pos))+1; dl=rng.poisson(depth,len(pos))+1; fh=rng.binomial(dh,p_hi)/dh; fl=rng.binomial(dl,p_lo)/dl; delta=fh-fl; k=max(1,int(window/20000)); sm=pd.Series(delta).rolling(k,center=True,min_periods=max(1,k//3)).mean().to_numpy(); ok=~np.isnan(sm); abs_sm=np.abs(sm); peak_i=int(np.nanargmax(abs_sm)); peak=pos[peak_i]; cutoff=.5*abs_sm[peak_i]; left=peak_i; right=peak_i
 while left>0 and np.isfinite(abs_sm[left-1]) and abs_sm[left-1]>=cutoff: left-=1
 while right<len(pos)-1 and np.isfinite(abs_sm[right+1]) and abs_sm[right+1]>=cutoff: right+=1
 interval_width=float(pos[right]-pos[left]); noise=np.median(np.abs(sm[ok]-np.median(sm[ok])))+1e-6; return abs(peak-qtl),interval_width,float(abs_sm[peak_i]/noise)
rows=[]
for N in [20,50,120]:
 for d in [15,40,100]:
  for w in [100000,300000,700000]:
   vals=np.array([one(N,d,w) for _ in range(20)]); rows.append((N,d,w,*vals.mean(axis=0)))
res=pd.DataFrame(rows,columns=['pool_n','depth','window','mean_peak_error','mean_halfmax_width','mean_SNR']); print(res.round(2).to_string(index=False))


 pool_n  depth  window  mean_peak_error  mean_halfmax_width  mean_SNR
     20     15  100000          97000.0            772000.0      8.37
     20     15  300000          64000.0           1032000.0     12.19
     20     15  700000          41000.0           1105000.0     16.56
     20     40  100000          88000.0            922000.0     10.19
     20     40  300000          67000.0           1029000.0     15.10
     20     40  700000          57000.0           1166000.0     19.08
     20    100  100000          75000.0            876000.0     12.49
     20    100  300000          68000.0           1059000.0     17.76
     20    100  700000          55000.0           1182000.0     24.53
     50     15  100000         134000.0            792000.0      9.24
     50     15  300000          75000.0           1002000.0     13.08
     50     15  700000          47000.0           1155000.0     18.76
     50     40  100000          78000.0            935000.0     11.89
     50     40  3000

结果解释：更大窗口通常提高本例信噪指标（signal-to-noise ratio, SNR）并增大半峰高宽度，深度和池大小影响不同抽样噪声；但mean_peak_error并非每一行都单调，因为有限重复仍有随机波动。本模拟逐位点生成池频率且不显式模拟染色体重组，因此只能拆解抽样与平滑，不可用来给真实BSA设计提供固定最优参数。

可修改练习：把每个条件的重复次数提高5倍并画出误差分布；加入一个相邻第二QTL，观察宽窗口何时把两个峰合并。